In [1]:
!nvidia-smi

Fri Oct  9 04:56:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

from numba import cuda

device = cuda.get_current_device()

print("Device Name:", device.name)
print("Compute Capability:", device.compute_capability)
print("Multiprocessor Count:", device.MULTIPROCESSOR_COUNT)
print("Max Threads Per Block:", device.MAX_THREADS_PER_BLOCK)
print("Warp Size:", device.WARP_SIZE)

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count: 40
Max Threads Per Block: 1024
Warp Size: 32


In [3]:

import numpy as np
import time
from numba import cuda

#1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

#2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [9]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)

    h_in = np.random.rand(N).astype(np.float32)

    # CPU Pure Loop
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(
        np.ones(10, dtype=np.float32)
    )

    double_gpu_kernel[1, 32](d_warmup, d_warmup)

    cuda.synchronize()

    # GPU Roundtrip Measurement
    t0 = time.perf_counter()

    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)

    double_gpu_kernel[
        blocks_per_grid,
        threads_per_block
    ](d_in, d_out)

    h_out = d_out.copy_to_host()

    cuda.synchronize()

    gpu_total_ms = (
        time.perf_counter() - t0
    ) * 1000.0

    # GPU Compute-Only Measurement
    t0 = time.perf_counter()

    double_gpu_kernel[
        blocks_per_grid,
        threads_per_block
    ](d_in, d_out)

    cuda.synchronize()

    gpu_compute_ms = (
        time.perf_counter() - t0
    ) * 1000.0

    return (
        cpu_time_ms,
        gpu_total_ms,
        gpu_compute_ms
    )

In [10]:
sizes = [
    100,
    1_000,
    10_000,
    100_000,
    1_000_000,
    5_000_000,
    10_000_000
]

results = []

print(f"{'N':>12} | {'CPU Time (ms)':>15} | {'GPU Total (ms)':>15} | {'GPU Kernel (ms)':>17} | Winner")
print("-" * 85)

for N in sizes:
    cpu, gpu_total, gpu_kernel = benchmark_run(N)

    if np.isnan(cpu):
        cpu_str = "N/A"
        winner = "GPU"
    else:
        cpu_str = f"{cpu:.3f}"
        winner = "CPU" if cpu < gpu_total else "GPU"

    print(
        f"{N:>12} | "
        f"{cpu_str:>15} | "
        f"{gpu_total:>15.3f} | "
        f"{gpu_kernel:>17.3f} | "
        f"{winner}"
    )

    results.append(
        (N, cpu, gpu_total, gpu_kernel, winner)
    )

           N |   CPU Time (ms) |  GPU Total (ms) |   GPU Kernel (ms) | Winner
-------------------------------------------------------------------------------------
         100 |           0.045 |           1.767 |             0.130 | CPU
        1000 |           0.280 |           0.749 |             0.128 | CPU
       10000 |           3.274 |           0.449 |             0.061 | GPU
      100000 |          26.770 |           0.627 |             0.068 | GPU
     1000000 |         270.068 |           4.725 |             0.113 | GPU
     5000000 |             N/A |          15.797 |             0.311 | GPU
    10000000 |             N/A |          25.108 |             0.452 | GPU


In [6]:

import numpy as np
from numba import cuda

@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")

defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()

print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print("Elements processed:", len(h_result))
print("Last element:", h_result[-1])

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000
Last element: 999.0


In [7]:

@cuda.jit
def inspect_threads_kernel(
    log_block, log_thread, log_global, log_active, N
):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0


N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](
    log_b, log_t, log_g, log_a, N
)
cuda.synchronize()

log_b = log_b.copy_to_host()
log_t = log_t.copy_to_host()
log_g = log_g.copy_to_host()
log_a = log_a.copy_to_host()

print("Global ID | Block ID | Thread ID | Status")
for i in range(32, 40):
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(log_g[i], log_b[i], log_t[i], status)

Global ID | Block ID | Thread ID | Status
32 4 0 ACTIVE
33 4 1 ACTIVE
34 4 2 ACTIVE
35 4 3 ACTIVE
36 4 4 ACTIVE
37 4 5 IDLE (GUARDED)
38 4 6 IDLE (GUARDED)
39 4 7 IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:

import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

print("CUDA kernels defined successfully")

CUDA kernels defined successfully


In [11]:

import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

print("CUDA kernels defined successfully")

CUDA kernels defined successfully


In [12]:

N = 50_000

h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads = 256
blocks = (N + threads - 1) // threads

print("Expected sum:", N)
print("Blocks:", blocks)
print("Threads per block:", threads)
print()

naive_results = []

for trial in range(1, 6):
    d_total = cuda.to_device(
        np.zeros(1, dtype=np.float32)
    )

    start = time.perf_counter()

    naive_sum_kernel[blocks, threads](d_arr, d_total)
    cuda.synchronize()

    elapsed_ms = (time.perf_counter() - start) * 1000
    actual = float(d_total.copy_to_host()[0])
    deviation = N - actual

    naive_results.append((actual, elapsed_ms))

    print(
        f"Trial {trial}: Expected={N}, "
        f"Actual={actual:.0f}, "
        f"Deviation={deviation:.0f}, "
        f"Time={elapsed_ms:.3f} ms"
    )

Expected sum: 50000
Blocks: 196
Threads per block: 256

Trial 1: Expected=50000, Actual=2, Deviation=49998, Time=60.825 ms
Trial 2: Expected=50000, Actual=2, Deviation=49998, Time=0.159 ms
Trial 3: Expected=50000, Actual=2, Deviation=49998, Time=0.137 ms
Trial 4: Expected=50000, Actual=2, Deviation=49998, Time=0.134 ms
Trial 5: Expected=50000, Actual=2, Deviation=49998, Time=0.113 ms


In [13]:

atomic_results = []

for trial in range(1, 6):
    d_total = cuda.to_device(
        np.zeros(1, dtype=np.float32)
    )

    start = time.perf_counter()

    atomic_sum_kernel[blocks, threads](d_arr, d_total)
    cuda.synchronize()

    elapsed_ms = (time.perf_counter() - start) * 1000
    actual = float(d_total.copy_to_host()[0])

    atomic_results.append((actual, elapsed_ms))

    print(
        f"Trial {trial}: Sum={actual:.0f}, "
        f"Correct={actual == N}, "
        f"Time={elapsed_ms:.3f} ms"
    )

Trial 1: Sum=50000, Correct=True, Time=144.711 ms
Trial 2: Sum=50000, Correct=True, Time=0.409 ms
Trial 3: Sum=50000, Correct=True, Time=0.387 ms
Trial 4: Sum=50000, Correct=True, Time=0.363 ms
Trial 5: Sum=50000, Correct=True, Time=0.371 ms


In [14]:

naive_avg = np.mean([t for _, t in naive_results])
atomic_avg = np.mean([t for _, t in atomic_results])

print(f"Naive average time:  {naive_avg:.3f} ms")
print(f"Atomic average time: {atomic_avg:.3f} ms")
print(f"Atomic / Naive time: {atomic_avg / naive_avg:.2f}x")

Naive average time:  12.274 ms
Atomic average time: 29.248 ms
Atomic / Naive time: 2.38x


In [15]:

import numpy as np
import math
from numba import cuda

@cuda.jit
def radial_vignette_2d(
    d_canvas, width, height, center_x, center_y
):
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(
            float(center_x**2 + center_y**2)
        )

        intensity = 1.0 - dist / max_dist

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity

print("2D CUDA kernel defined successfully")

2D CUDA kernel defined successfully


In [16]:

WIDTH = 1024
HEIGHT = 1024

# Create an empty image matrix
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# Configure the 2D thread geometry
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + 15) // 16,
    (HEIGHT + 15) // 16
)

# Run the CUDA kernel
radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas,
    WIDTH,
    HEIGHT,
    WIDTH // 2,
    HEIGHT // 2
)

cuda.synchronize()

# Copy the result back to the CPU
result_canvas = d_canvas.copy_to_host()

print("Matrix shape:", result_canvas.shape)
print("Center pixel:", result_canvas[512, 512])
print("Corner pixel:", result_canvas[0, 0])
print("Blocks in X:", blocks_2d[0])
print("Blocks in Y:", blocks_2d[1])
print("Total blocks:", blocks_2d[0] * blocks_2d[1])

Matrix shape: (1024, 1024)
Center pixel: 1.0
Corner pixel: 0.0
Blocks in X: 64
Blocks in Y: 64
Total blocks: 4096


In [17]:

import hashlib
from numba import cuda

STUDENT_ID = "230103161"
CROSSOVER_N = 10000

def generate_lab_checksum(student_id, d_canvas, crossover_n):
    h = hashlib.sha256()

    h.update(student_id.strip().encode("utf-8"))
    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode("utf-8")
    h.update(gpu_name)

    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(crossover_n).encode("utf-8"))

    token = h.hexdigest()[:16].upper()

    print("=" * 50)
    print("FINAL SUBMISSION TOKEN:", token)
    print("=" * 50)

    return token

token = generate_lab_checksum(
    STUDENT_ID, d_canvas, CROSSOVER_N
)

FINAL SUBMISSION TOKEN: 53ABFF0A016F45ED
